# Step 3b: Negative Intent Volume Analysis

This notebook combines saved search-term metrics with candidate search volumes to produce intent-level, endpoint-level weighted violation rates and summaries.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Configuration
VARIANT = 'control'
DEFAULT_K = 10
K_VALUES = [4, 10, 36]
METRICS_DIR = f'./metrics_output/{VARIANT}'
SEARCH_TERM_METRICS_FILE = os.path.join(METRICS_DIR, f'{VARIANT}_search_term_metrics.csv')

print(f"Variant: {VARIANT}")
print(f"Search-term metrics: {SEARCH_TERM_METRICS_FILE}")
print(f"Default K: {DEFAULT_K}, K values: {K_VALUES}")

Variant: control
Search-term metrics: ./metrics_output/control/control_search_term_metrics.csv
Default K: 10, K values: [4, 10, 36]


In [3]:
# Load search-term metrics (saved in Step 3a with search_volume already included)
df_metrics = pd.read_csv(SEARCH_TERM_METRICS_FILE)
df_metrics['k'] = pd.to_numeric(df_metrics['k'], errors='coerce')
df_metrics['search_volume'] = pd.to_numeric(df_metrics['search_volume'], errors='coerce').fillna(0)
df_metrics['has_volume'] = df_metrics['search_volume'] > 0
print(f"Loaded metrics: {df_metrics.shape} (rows, cols)")
print(df_metrics.head())

# Coverage stats
total_terms = df_metrics['search_term'].nunique()
terms_with_vol = df_metrics[df_metrics['has_volume']]['search_term'].nunique()
pct = (terms_with_vol/total_terms*100 if total_terms else 0)
print(f"Volume coverage: {terms_with_vol}/{total_terms} search terms ({pct:.1f}%)")

Loaded metrics: (9534, 14) (rows, cols)
   variant   endpoint  k                                  search_term  \
0  control  l1_hybrid  4  2 hounds design freedom no pull dog harness   
1  control  l1_hybrid  4                             acana grain free   
2  control  l1_hybrid  4                         acana grain free cat   
3  control  l1_hybrid  4               acana grain-free food for dogs   
4  control  l1_hybrid  4                   adjustable no pull harness   

   violation_rate  compliant_rate  unclear_rate  total_results  \
0            0.75            0.00          0.25              4   
1            0.25            0.75          0.00              4   
2            0.00            0.75          0.25              4   
3            0.25            0.75          0.00              4   
4            0.50            0.00          0.50              4   

   violation_count  compliant_count  unclear_count negative_intent  \
0                3                0              1    

In [4]:
# Prepare data for analysis
df_merged = df_metrics.copy()
# Ensure search_volume is numeric
if 'search_volume' in df_merged.columns:
    df_merged['search_volume'] = pd.to_numeric(df_merged['search_volume'], errors='coerce').fillna(0)
else:
    df_merged['search_volume'] = 0
df_merged['has_volume'] = df_merged['search_volume'] > 0

print(f"Processing dataset: {df_merged.shape} (rows, cols)")
print(df_merged.head())

Processing dataset: (9534, 14) (rows, cols)
   variant   endpoint  k                                  search_term  \
0  control  l1_hybrid  4  2 hounds design freedom no pull dog harness   
1  control  l1_hybrid  4                             acana grain free   
2  control  l1_hybrid  4                         acana grain free cat   
3  control  l1_hybrid  4               acana grain-free food for dogs   
4  control  l1_hybrid  4                   adjustable no pull harness   

   violation_rate  compliant_rate  unclear_rate  total_results  \
0            0.75            0.00          0.25              4   
1            0.25            0.75          0.00              4   
2            0.00            0.75          0.25              4   
3            0.25            0.75          0.00              4   
4            0.50            0.00          0.50              4   

   violation_count  compliant_count  unclear_count negative_intent  \
0                3                0              1

In [5]:
df_merged.head()

,variant,endpoint,k,search_term,violation_rate,compliant_rate,unclear_rate,total_results,violation_count,compliant_count,unclear_count,negative_intent,search_volume,has_volume
0,control,l1_hybrid,4,2 hounds design freedom no pull dog harness,0.75,0.00,0.25,4,3,0,1,pull,13268,True
1,control,l1_hybrid,4,acana grain free,0.25,0.75,0.00,4,1,3,0,grain,1372,True
2,control,l1_hybrid,4,acana grain free cat,0.00,0.75,0.25,4,0,3,1,grain,1699,True
3,control,l1_hybrid,4,acana grain-free food for dogs,0.25,0.75,0.00,4,1,3,0,grain,3087,True
4,control,l1_hybrid,4,adjustable no pull harness,0.50,0.00,0.50,4,2,0,2,pull,974,True


In [6]:
df_merged.isnull().sum()

variant            0
endpoint           0
k                  0
search_term        0
violation_rate     0
compliant_rate     0
unclear_rate       0
total_results      0
violation_count    0
compliant_count    0
unclear_count      0
negative_intent    0
search_volume      0
has_volume         0
dtype: int64

In [7]:
df_merged.negative_intent.value_counts()

negative_intent
grain      3984
chicken     708
pull        636
rawhide     564
hide        432
           ... 
grow         12
hides        12
holes        12
lick         12
yeast        12
Name: count, Length: 102, dtype: int64

In [8]:
# Compute weighted violation metrics by endpoint/k and by negative intent
def weighted_avg(series, weights):
    w = pd.to_numeric(weights, errors='coerce').fillna(0)
    s = pd.to_numeric(series, errors='coerce').fillna(0)
    tot = w.sum()
    return (s.mul(w).sum() / tot) if tot > 0 else np.nan

summary_rows = []
if not df_merged.empty:
    # Filter to terms with volume
    dfv = df_merged[df_merged['has_volume']].copy()
    # Overall (no intent), and per intent
    group_cols_base = ['endpoint', 'k']
    for with_intent in [False, True]:
        group_cols = group_cols_base + (['negative_intent'] if with_intent else [])
        grp = dfv.groupby(group_cols)
        for keys, sub in grp:
            row = {c: v for c, v in zip(group_cols, keys if isinstance(keys, tuple) else (keys,))}
            row['variant'] = VARIANT
            if not with_intent:
                row['negative_intent'] = 'all negative intent'
            row['terms_with_volume'] = sub['search_term'].nunique()
            row['total_volume'] = pd.to_numeric(sub['search_volume'], errors='coerce').sum()
            # Weighted averages of rates
            row['weighted_violation_rate'] = weighted_avg(sub['violation_rate'], sub['search_volume'])
            row['weighted_compliant_rate'] = weighted_avg(sub['compliant_rate'], sub['search_volume'])
            row['weighted_unclear_rate'] = weighted_avg(sub['unclear_rate'], sub['search_volume'])
            summary_rows.append(row)
    df_summary = pd.DataFrame(summary_rows)
    df_summary = df_summary.sort_values(['endpoint', 'k', 'negative_intent']).reset_index(drop=True)
    print(f"Summary rows: {len(df_summary)}")
    print(df_summary.head())
else:
    df_summary = pd.DataFrame()
    print("No data available to compute weighted metrics.")

df_summary = df_summary.sort_values(['endpoint', 'k', 'terms_with_volume'], ascending=False).reset_index(drop=True)

Summary rows: 1236
  endpoint  k  variant      negative_intent  terms_with_volume  total_volume  \
0      knn  4  control            additives                  1          1600   
1      knn  4  control  all negative intent                795       7015957   
2      knn  4  control                aller                  1           974   
3      knn  4  control              allergy                  3          8756   
4      knn  4  control                 bark                  1         10077   

   weighted_violation_rate  weighted_compliant_rate  weighted_unclear_rate  
0                 0.500000                 0.000000               0.500000  
1                 0.164927                 0.765797               0.069276  
2                 0.500000                 0.500000               0.000000  
3                 0.179705                 0.570694               0.249600  
4                 0.500000                 0.500000               0.000000  


In [9]:
df_summary.to_csv(os.path.join(METRICS_DIR, f'{VARIANT}_negative_intent_volume_analysis.csv'), index=False)

In [10]:
df_summary.head()

,endpoint,k,variant,negative_intent,terms_with_volume,total_volume,weighted_violation_rate,weighted_compliant_rate,weighted_unclear_rate
0,lexical,36,control,all negative intent,793,7012954,0.228046,0.705015,0.066939
1,lexical,36,control,grain,332,2692575,0.060697,0.922508,0.016795
2,lexical,36,control,chicken,59,461986,0.819146,0.152885,0.027969
3,lexical,36,control,pull,53,820714,0.258814,0.730594,0.010593
4,lexical,36,control,rawhide,47,760133,0.111319,0.813080,0.075601


## quick view 

### rates for each endpoint + k (all intents combined)

In [11]:
# View endpoint/k level aggregation (all intents combined)
df_endpoint_k = df_summary[df_summary['negative_intent'] == 'all negative intent'].round(4).copy()
print(f"Endpoint/K level metrics (all intents combined): {len(df_endpoint_k)} rows")
df_endpoint_k.sort_values(['k', 'endpoint'], inplace=True)

df_endpoint_k

Endpoint/K level metrics (all intents combined): 12 rows


,endpoint,k,variant,negative_intent,terms_with_volume,total_volume,weighted_violation_rate,weighted_compliant_rate,weighted_unclear_rate
1133,knn,4,control,all negative intent,795,7015957,0.1649,0.7658,0.0693
824,l1_hybrid,4,control,all negative intent,795,7015957,0.1880,0.7603,0.0517
515,l2,4,control,all negative intent,795,7015957,0.1893,0.7559,0.0548
206,lexical,4,control,all negative intent,793,7012954,0.1990,0.7706,0.0304
1030,knn,10,control,all negative intent,795,7015957,0.1644,0.7629,0.0727
721,l1_hybrid,10,control,all negative intent,795,7015957,0.1970,0.7467,0.0563
412,l2,10,control,all negative intent,795,7015957,0.1945,0.7507,0.0548
103,lexical,10,control,all negative intent,793,7012954,0.2208,0.7359,0.0433
927,knn,36,control,all negative intent,795,7015957,0.1737,0.7418,0.0844
618,l1_hybrid,36,control,all negative intent,795,7015957,0.2000,0.7250,0.0750


## rates for some top negative intent for each endpoint at DEFAULT_K

### KNN 

In [12]:
df_summary_at_knn = df_summary[(df_summary['endpoint'] == 'knn') & (df_summary['k'] == DEFAULT_K) & (df_summary['terms_with_volume'] > 8)].copy()
df_summary_at_knn


,endpoint,k,variant,negative_intent,terms_with_volume,total_volume,weighted_violation_rate,weighted_compliant_rate,weighted_unclear_rate
1030,knn,10,control,all negative intent,795,7015957,0.164410,0.762859,0.072731
1031,knn,10,control,grain,332,2692575,0.058003,0.898236,0.043761
1032,knn,10,control,chicken,59,461986,0.471303,0.266846,0.261851
1033,knn,10,control,pull,53,820714,0.120310,0.878635,0.001055
1034,knn,10,control,rawhide,47,760133,0.071912,0.904376,0.023712
1035,knn,10,control,hide,36,388507,0.058502,0.941206,0.000292
1036,knn,10,control,stuffing,24,369948,0.000546,0.999454,0.000000
1037,knn,10,control,dust,14,170435,0.033324,0.201260,0.765416
1038,knn,10,control,mess,14,59732,0.004205,0.995795,0.000000
1039,knn,10,control,gluten,10,66998,0.006073,0.737365,0.256561


In [13]:
# Show sample search terms for each negative intent in df_summary_at_knn
print("At L1 knn level, Sample search terms by negative intent:")
print("=" * 80)

# Calculate average weighted violation rate (excluding 'all negative intent')
all_intents = df_summary_at_knn[df_summary_at_knn['negative_intent'] == 'all negative intent']
avg_violation_rate = all_intents['weighted_violation_rate'].mean()
print(f"\nAverage weighted violation rate for all negative intent: {avg_violation_rate:.4f}")
print("=" * 80)

for idx, row in df_summary_at_knn.iterrows():
    intent = row['negative_intent']
    endpoint = row['endpoint']
    k = row['k']
    
    # Skip the 'all negative intent' aggregated row
    if intent == 'all negative intent':
        continue
    
    # Get search terms for this intent from the original data, sorted by search volume
    matching_data = df_merged[
        (df_merged['negative_intent'] == intent) & 
        (df_merged['has_volume'] == True) &
        (df_merged['endpoint'] == endpoint) &
        (df_merged['k'] == k)
    ][['search_term', 'search_volume', 'violation_rate']].drop_duplicates('search_term').sort_values('search_volume', ascending=False)
    
    violation_rate = row['weighted_violation_rate']
    comparison = "above average" if violation_rate > avg_violation_rate else "below average" if violation_rate < avg_violation_rate else "at average"
    
    print(f"\n{intent}")
    print(f"  Terms with volume: {len(matching_data)}")
    print(f"  Weighted violation rate: {violation_rate:.4f} ({comparison})")
    print(f"  Top 5 terms by volume:")
    for i, (_, term_row) in enumerate(matching_data.head(5).iterrows(), 1):
        print(f"    {i}. {term_row['search_term']} (volume: {term_row['search_volume']:.0f}, violation rate: {term_row['violation_rate']:.4f})")
    if len(matching_data) > 5:
        print(f"  ... and {len(matching_data) - 5} more")

At L1 knn level, Sample search terms by negative intent:

Average weighted violation rate for all negative intent: 0.1644

grain
  Terms with volume: 332
  Weighted violation rate: 0.0580 (below average)
  Top 5 terms by volume:
    1. grain free dog food (volume: 611730, violation rate: 0.0000)
    2. grain free dry cat food (volume: 197068, violation rate: 0.0000)
    3. grain free wet cat food (volume: 162972, violation rate: 0.0000)
    4. grain free dog treats (volume: 138329, violation rate: 0.0000)
    5. grain free cat food (volume: 76423, violation rate: 0.0000)
  ... and 327 more

chicken
  Terms with volume: 59
  Weighted violation rate: 0.4713 (above average)
  Top 5 terms by volume:
    1. chicken free dry dog food (volume: 124263, violation rate: 0.5000)
    2. chicken free dog treats (volume: 68590, violation rate: 0.2000)
    3. chicken free dog food (volume: 38479, violation rate: 0.3000)
    4. no chicken dog food (volume: 32850, violation rate: 0.5000)
    5. chicken

### Lexical 

In [14]:
df_summary_at_lexical = df_summary[(df_summary['endpoint'] == 'lexical') & (df_summary['k'] == DEFAULT_K) & (df_summary['terms_with_volume'] > 8)].copy()
df_summary_at_lexical

,endpoint,k,variant,negative_intent,terms_with_volume,total_volume,weighted_violation_rate,weighted_compliant_rate,weighted_unclear_rate
103,lexical,10,control,all negative intent,793,7012954,0.220813,0.735900,0.043287
104,lexical,10,control,grain,332,2692575,0.060782,0.931907,0.007311
105,lexical,10,control,chicken,59,461986,0.801859,0.177908,0.020233
106,lexical,10,control,pull,53,820714,0.158757,0.833695,0.007548
107,lexical,10,control,rawhide,47,760133,0.083027,0.871763,0.045209
108,lexical,10,control,hide,36,388507,0.243234,0.755354,0.001412
109,lexical,10,control,stuffing,24,369948,0.001638,0.998362,0.000000
110,lexical,10,control,dust,14,170435,0.197884,0.267990,0.534127
111,lexical,10,control,mess,14,59732,0.055367,0.662407,0.282226
112,lexical,10,control,gluten,10,66998,0.324062,0.653481,0.022457


In [15]:
# Show sample search terms for each negative intent in df_summary_at_lexical
print("At L1 lexical level, Sample search terms by negative intent:")
print("=" * 80)

# Calculate average weighted violation rate (excluding 'all negative intent')
all_intents = df_summary_at_lexical[df_summary_at_lexical['negative_intent'] == 'all negative intent']
avg_violation_rate = all_intents['weighted_violation_rate'].mean()
print(f"\nAverage weighted violation rate for all negative intent: {avg_violation_rate:.4f}")
print("=" * 80)

for idx, row in df_summary_at_lexical.iterrows():
    intent = row['negative_intent']
    endpoint = row['endpoint']
    k = row['k']
    
    # Skip the 'all negative intent' aggregated row
    if intent == 'all negative intent':
        continue
    
    # Get search terms for this intent from the original data, sorted by search volume
    matching_data = df_merged[
        (df_merged['negative_intent'] == intent) & 
        (df_merged['has_volume'] == True) &
        (df_merged['endpoint'] == endpoint) &
        (df_merged['k'] == k)
    ][['search_term', 'search_volume', 'violation_rate']].drop_duplicates('search_term').sort_values('search_volume', ascending=False)
    
    violation_rate = row['weighted_violation_rate']
    comparison = "above average" if violation_rate > avg_violation_rate else "below average" if violation_rate < avg_violation_rate else "at average"
    
    print(f"\n{intent}")
    print(f"  Terms with volume: {len(matching_data)}")
    print(f"  Weighted violation rate: {violation_rate:.4f} ({comparison})")
    print(f"  Top 5 terms by volume:")
    for i, (_, term_row) in enumerate(matching_data.head(5).iterrows(), 1):
        print(f"    {i}. {term_row['search_term']} (volume: {term_row['search_volume']:.0f}, violation rate: {term_row['violation_rate']:.4f})")
    if len(matching_data) > 5:
        print(f"  ... and {len(matching_data) - 5} more")

At L1 lexical level, Sample search terms by negative intent:

Average weighted violation rate for all negative intent: 0.2208

grain
  Terms with volume: 332
  Weighted violation rate: 0.0608 (below average)
  Top 5 terms by volume:
    1. grain free dog food (volume: 611730, violation rate: 0.1000)
    2. grain free dry cat food (volume: 197068, violation rate: 0.0000)
    3. grain free wet cat food (volume: 162972, violation rate: 0.0000)
    4. grain free dog treats (volume: 138329, violation rate: 0.0000)
    5. grain free cat food (volume: 76423, violation rate: 0.0000)
  ... and 327 more

chicken
  Terms with volume: 59
  Weighted violation rate: 0.8019 (above average)
  Top 5 terms by volume:
    1. chicken free dry dog food (volume: 124263, violation rate: 0.8000)
    2. chicken free dog treats (volume: 68590, violation rate: 0.8000)
    3. chicken free dog food (volume: 38479, violation rate: 0.7000)
    4. no chicken dog food (volume: 32850, violation rate: 1.0000)
    5. chi

### l1 hybrid 

In [16]:
df_summary_at_l1_hybrid = df_summary[(df_summary['endpoint'] == 'l1_hybrid') & (df_summary['k'] == DEFAULT_K) & (df_summary['terms_with_volume'] > 8)].copy()
df_summary_at_l1_hybrid

,endpoint,k,variant,negative_intent,terms_with_volume,total_volume,weighted_violation_rate,weighted_compliant_rate,weighted_unclear_rate
721,l1_hybrid,10,control,all negative intent,795,7015957,0.196991,0.746697,0.056311
722,l1_hybrid,10,control,grain,332,2692575,0.072152,0.908507,0.019341
723,l1_hybrid,10,control,chicken,59,461986,0.722533,0.166898,0.110569
724,l1_hybrid,10,control,pull,53,820714,0.162804,0.830579,0.006616
725,l1_hybrid,10,control,rawhide,47,760133,0.162173,0.811145,0.026682
726,l1_hybrid,10,control,hide,36,388507,0.065221,0.934487,0.000292
727,l1_hybrid,10,control,stuffing,24,369948,0.004272,0.995728,0.000000
728,l1_hybrid,10,control,dust,14,170435,0.029708,0.210298,0.759995
729,l1_hybrid,10,control,mess,14,59732,0.020587,0.677680,0.301733
730,l1_hybrid,10,control,gluten,10,66998,0.210193,0.681977,0.107830


In [17]:
# Show sample search terms for each negative intent in df_summary_at_l1_hybrid
print("At L1 hybrid level, Sample search terms by negative intent:")
print("=" * 80)

# Calculate average weighted violation rate (excluding 'all negative intent')
all_intents = df_summary_at_l1_hybrid[df_summary_at_l1_hybrid['negative_intent'] == 'all negative intent']
avg_violation_rate = all_intents['weighted_violation_rate'].mean()
print(f"\nAverage weighted violation rate for all negative intent: {avg_violation_rate:.4f}")
print("=" * 80)

for idx, row in df_summary_at_l1_hybrid.iterrows():
    intent = row['negative_intent']
    endpoint = row['endpoint']
    k = row['k']
    
    # Skip the 'all negative intent' aggregated row
    if intent == 'all negative intent':
        continue
    
    # Get search terms for this intent from the original data, sorted by search volume
    matching_data = df_merged[
        (df_merged['negative_intent'] == intent) & 
        (df_merged['has_volume'] == True) &
        (df_merged['endpoint'] == endpoint) &
        (df_merged['k'] == k)
    ][['search_term', 'search_volume', 'violation_rate']].drop_duplicates('search_term').sort_values('search_volume', ascending=False)
    
    violation_rate = row['weighted_violation_rate']
    comparison = "above average" if violation_rate > avg_violation_rate else "below average" if violation_rate < avg_violation_rate else "at average"
    
    print(f"\n{intent}")
    print(f"  Terms with volume: {len(matching_data)}")
    print(f"  Weighted violation rate: {violation_rate:.4f} ({comparison})")
    print(f"  Top 5 terms by volume:")
    for i, (_, term_row) in enumerate(matching_data.head(5).iterrows(), 1):
        print(f"    {i}. {term_row['search_term']} (volume: {term_row['search_volume']:.0f}, violation rate: {term_row['violation_rate']:.4f})")
    if len(matching_data) > 5:
        print(f"  ... and {len(matching_data) - 5} more")

At L1 hybrid level, Sample search terms by negative intent:

Average weighted violation rate for all negative intent: 0.1970

grain
  Terms with volume: 332
  Weighted violation rate: 0.0722 (below average)
  Top 5 terms by volume:
    1. grain free dog food (volume: 611730, violation rate: 0.0000)
    2. grain free dry cat food (volume: 197068, violation rate: 0.0000)
    3. grain free wet cat food (volume: 162972, violation rate: 0.0000)
    4. grain free dog treats (volume: 138329, violation rate: 0.1000)
    5. grain free cat food (volume: 76423, violation rate: 0.0000)
  ... and 327 more

chicken
  Terms with volume: 59
  Weighted violation rate: 0.7225 (above average)
  Top 5 terms by volume:
    1. chicken free dry dog food (volume: 124263, violation rate: 0.9000)
    2. chicken free dog treats (volume: 68590, violation rate: 0.6000)
    3. chicken free dog food (volume: 38479, violation rate: 0.7000)
    4. no chicken dog food (volume: 32850, violation rate: 0.5000)
    5. chic

### L2 level

In [18]:
df_summary_at_l2 = df_summary[(df_summary['endpoint'] == 'l2') & (df_summary['k'] == DEFAULT_K) & (df_summary['terms_with_volume'] > 8)].copy()
df_summary_at_l2

,endpoint,k,variant,negative_intent,terms_with_volume,total_volume,weighted_violation_rate,weighted_compliant_rate,weighted_unclear_rate
412,l2,10,control,all negative intent,795,7015957,0.194483,0.750697,0.054820
413,l2,10,control,grain,332,2692575,0.071003,0.910799,0.018198
414,l2,10,control,chicken,59,461986,0.713226,0.175337,0.111437
415,l2,10,control,pull,53,820714,0.162509,0.829982,0.007508
416,l2,10,control,rawhide,47,760133,0.166506,0.807839,0.025655
417,l2,10,control,hide,36,388507,0.063970,0.935738,0.000292
418,l2,10,control,stuffing,24,369948,0.004272,0.995728,0.000000
419,l2,10,control,dust,14,170435,0.029708,0.252868,0.717424
420,l2,10,control,mess,14,59732,0.022248,0.678469,0.299283
421,l2,10,control,gluten,10,66998,0.212724,0.681107,0.106169


In [19]:
# Show sample search terms for each negative intent in df_summary_at_l2
print("At L2 level, Sample search terms by negative intent:")
print("=" * 80)

# Calculate average weighted violation rate (excluding 'all negative intent')
all_intents = df_summary_at_l2[df_summary_at_l2['negative_intent'] == 'all negative intent']
avg_violation_rate = all_intents['weighted_violation_rate'].mean()
print(f"\nAverage weighted violation rate for all negative intent: {avg_violation_rate:.4f}")
print("=" * 80)

for idx, row in df_summary_at_l2.iterrows():
    intent = row['negative_intent']
    endpoint = row['endpoint']
    k = row['k']
    
    # Skip the 'all negative intent' aggregated row
    if intent == 'all negative intent':
        continue
    
    # Get search terms for this intent from the original data, sorted by search volume
    matching_data = df_merged[
        (df_merged['negative_intent'] == intent) & 
        (df_merged['has_volume'] == True) &
        (df_merged['endpoint'] == endpoint) &
        (df_merged['k'] == k)
    ][['search_term', 'search_volume', 'violation_rate']].drop_duplicates('search_term').sort_values('search_volume', ascending=False)
    
    violation_rate = row['weighted_violation_rate']
    comparison = "above average" if violation_rate > avg_violation_rate else "below average" if violation_rate < avg_violation_rate else "at average"
    
    print(f"\n{intent}")
    print(f"  Terms with volume: {len(matching_data)}")
    print(f"  Weighted violation rate: {violation_rate:.4f} ({comparison})")
    print(f"  Top 5 terms by volume:")
    for i, (_, term_row) in enumerate(matching_data.head(5).iterrows(), 1):
        print(f"    {i}. {term_row['search_term']} (volume: {term_row['search_volume']:.0f}, violation rate: {term_row['violation_rate']:.4f})")
    if len(matching_data) > 5:
        print(f"  ... and {len(matching_data) - 5} more")

At L2 level, Sample search terms by negative intent:

Average weighted violation rate for all negative intent: 0.1945

grain
  Terms with volume: 332
  Weighted violation rate: 0.0710 (below average)
  Top 5 terms by volume:
    1. grain free dog food (volume: 611730, violation rate: 0.0000)
    2. grain free dry cat food (volume: 197068, violation rate: 0.0000)
    3. grain free wet cat food (volume: 162972, violation rate: 0.0000)
    4. grain free dog treats (volume: 138329, violation rate: 0.1000)
    5. grain free cat food (volume: 76423, violation rate: 0.0000)
  ... and 327 more

chicken
  Terms with volume: 59
  Weighted violation rate: 0.7132 (above average)
  Top 5 terms by volume:
    1. chicken free dry dog food (volume: 124263, violation rate: 0.9000)
    2. chicken free dog treats (volume: 68590, violation rate: 0.6000)
    3. chicken free dog food (volume: 38479, violation rate: 0.6000)
    4. no chicken dog food (volume: 32850, violation rate: 0.5000)
    5. chicken fre